# Сравнение MLP и MGS-KAN по фолдам и seed

Выбери GPU и режим в первой ячейке. `smoke` проверяет обе модели на одном фолде за две эпохи. `full` запускает все доступные фолды и три пары seed за 50 эпох. Обучения идут последовательно.

В окружении ядра установи проект из корня репозитория командой `python -m pip install -e '.[tracking]' pandas`. Укажи `MLFLOW_TRACKING_URI` в корневом `.env` или окружении ядра. Первый датасет — Airfoil Self-Noise. Для первой загрузки в `data/` с официального источника выставь `DOWNLOAD = True`.

Если ядро запущено вне репозитория, укажи в первой ячейке абсолютный путь к копии проекта на сервере в `PROJECT_DIR`. При editable-установке корень определяется автоматически.

In [ ]:
PROJECT_DIR = None  # Например, "/home/user/mgs-kan"
MODE = "smoke"      # "smoke" или "full"
DEVICE = "cuda:0"   # Индекс среди GPU, видимых ядру. Для локальной проверки — "cpu".
DATASET = "airfoil"
VERSION = "v1"      # Измени для новой серии после изменения протокола.
DOWNLOAD = False

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

## План серии

В каждой паре первый seed задаёт начальные веса, второй — порядок батчей. Проверь размер серии и лимиты перед запуском следующего раздела.

In [ ]:
import pandas as pd
from IPython.display import display

from src.config import ModelSettings, MonitorSettings, ResourceSettings, TrackingSettings, TrainingSettings
from src.data import dataset_class
from src.experiments import ExperimentRunner

resources = ResourceSettings(_env_file=ENV_FILE)
models = ["mlp", "mgs_kan"]

supported_folds = dataset_class(DATASET).available_folds
if MODE == "smoke":
    folds = [supported_folds[0]]
    seeds = [(11, 101)]
    epochs = 2
    max_runs = 2
elif MODE == "full":
    folds = list(supported_folds)
    seeds = [(11, 101), (22, 202), (33, 303)]
    epochs = 50
    max_runs = 30
else:
    raise ValueError("Выбери MODE = 'smoke' или 'full'.")

series_id = f"{DATASET}-baselines-{MODE}-{VERSION}"
planned = len(models) * len(folds) * len(seeds)
print(f"{series_id} | {DEVICE} | {planned} обучений по {epochs} эпох")
print(f"Модели: {models} | Фолды: {folds} | Пары seed: {seeds}")
print(f"Потоков CPU на обучение: {resources.cpu_threads}")
if DEVICE.startswith("cuda:"):
    print(f"Лимит памяти PyTorch на выбранной GPU: {resources.gpu_memory_fraction:.0%}")

## Запуск и возобновление

После перезапуска ядра выполни ячейки с прежними параметрами. В том же эксперименте MLflow раннер найдёт завершённые комбинации по `series_id` и продолжит с оставшихся. После обновления кода алертов оставь прежнюю `VERSION`. Для отдельного сравнения на других данных или с другими настройками задай новую версию. Для проверки возобновления повтори `smoke` с той же версией и сравни `run_id` в таблице.

In [ ]:
runner = ExperimentRunner(
    training=TrainingSettings(epochs=epochs, _env_file=ENV_FILE),
    model_settings=ModelSettings(_env_file=ENV_FILE),
    resources=resources,
    tracking=TrackingSettings(_env_file=ENV_FILE),
    monitoring=MonitorSettings(_env_file=ENV_FILE),
)
summary = runner.run(
    series_id=series_id,
    dataset=DATASET,
    root=DATA_DIR,
    models=models,
    folds=folds,
    seeds=seeds,
    device=DEVICE,
    download=DOWNLOAD,
    max_runs=max_runs,
)

## Результаты по обучениям

RMSE и R² рассчитаны на тестовом фолде с весами лучшей эпохи. Пик памяти показывает выделения PyTorch в MiB и доступен при запуске на CUDA.

In [ ]:
from dataclasses import asdict

rows = []
for result in summary.results:
    metrics = runner.tracker.client.get_run(result.run_id).data.metrics
    peak_bytes = metrics.get("cuda_peak_allocated_bytes")
    rows.append({
        **asdict(result.trial),
        "best_epoch": result.best_epoch,
        "test_rmse": result.test_rmse,
        "test_r2": result.test_r2,
        "duration_s": metrics.get("duration_seconds"),
        "peak_gpu_mib": peak_bytes / 2**20 if peak_bytes is not None else None,
        "run_id": result.run_id,
    })

display(pd.DataFrame(rows))

## Сводка по моделям

Сначала метрики усредняются по seed внутри каждого фолда, затем — по фолдам с равным весом. Столбцы `completed` и `planned` показывают покрытие серии.

In [ ]:
by_model = pd.DataFrame.from_dict(summary.by_model, orient="index")
display(by_model.rename_axis("model"))
print(f"Осталось обучений: {len(summary.pending)}")
if summary.pending:
    display(pd.DataFrame(asdict(trial) for trial in summary.pending))